# 02 · Crossover — a double dissociation (HepG2 vs K562)

Runs the **same** 33k benchmark with a **HepG2 (hepatic)** ChromBPNet model and compares
per-element AUROC to the K562 (erythroid) run from notebook 01. If K562 wins on
hematopoietic elements *and* HepG2 wins on hepatic ones, the signal is **cell-type-driven,
not a model artifact** — proven by intervention, with a bootstrap CI on the deltas.

In [ ]:
# --- Setup: clone RegLens from GitHub + install -----------------------------
# No zip upload — clones the public repo fresh, so it's always current. Re-running
# this cell fast-forwards to the latest commit on main.
import os
if not os.path.isdir("/content/RegLens/.git"):
    !git clone --depth 1 https://github.com/kpal002/RegLens.git /content/RegLens
%cd /content/RegLens
!git pull -q --ff-only 2>/dev/null || echo "(using the freshly cloned snapshot)"
!nvidia-smi -L || echo 'NO GPU — set Runtime > Change runtime type > GPU.'
!pip -q install 'tensorflow>=2.12' pyfaidx typer numpy matplotlib
!pip -q install -e .
!echo "RegLens @ $(git rev-parse --short HEAD) | cwd $(pwd)"

In [ ]:
# --- Reference genome (hg38) + pretrained K562 ChromBPNet model -------------
import glob, os
os.makedirs('/content/ref', exist_ok=True)
%cd /content/ref
# hg38 via the Broad public bucket: resolves on Colab, chr-named, ships its .fai.
!wget -c -O hg38.fa     https://storage.googleapis.com/gcp-public-data--broad-references/hg38/v0/Homo_sapiens_assembly38.fasta
!wget -c -O hg38.fa.fai https://storage.googleapis.com/gcp-public-data--broad-references/hg38/v0/Homo_sapiens_assembly38.fasta.fai
# ENCODE K562 ATAC ChromBPNet models (5 folds, bias-corrected).
![ -f ENCFF984RAF.tar.gz ] || wget -q -c -O ENCFF984RAF.tar.gz https://www.encodeproject.org/files/ENCFF984RAF/@@download/ENCFF984RAF.tar.gz
!mkdir -p encode_models && tar -xzf ENCFF984RAF.tar.gz -C encode_models
n = len(glob.glob('encode_models/**/*chrombpnet_nobias*.h5', recursive=True))
print(n, 'K562 folds | genome:', os.path.exists('hg38.fa'))
%cd /content/RegLens

In [ ]:
# --- Score the full benchmark with the HepG2 model --------------------------
import glob, time
from reglens.validation import load_labeled_variants, evaluate_batched
from reglens.tools.chrombpnet_score import ChromBPNetScorer, KerasChromBPNetBackend

# K562 per-element AUROC from notebook 01 (paste your run, or use the committed values):
K562 = {'SORT1.2':0.584,'FOXE1':0.430,'SORT1-flip':0.638,'SORT1':0.586,'BCL11A':0.620,
'ZFAND3':0.637,'MYCrs6983267':0.634,'UC88':0.709,'MSMB':0.606,'TCF7L2':0.487,'RET':0.546,
'IRF6':0.533,'ZRSh-13':0.538,'MYCrs11986220':0.463,'PKLR-24h':0.794,'IRF4':0.548,
'ZRSh-13h2':0.564,'PKLR-48h':0.805,'GP1BA':0.729,'LDLR.2':0.679,'LDLR':0.691,'F9':0.642,
'HNF4A':0.611,'HBG1':0.663,'TERT-GSc':0.672,'TERT-GBM':0.672,'TERT-GAa':0.649,
'TERT-HEK':0.705,'HBB':0.684}

# HepG2 (hepatic) ATAC ChromBPNet model — ENCODE ENCFF137WCM (5 folds):
if not glob.glob('/content/ref/hepg2_models/**/*chrombpnet_nobias*.h5', recursive=True):
    !cd /content/ref && wget -q -c -O ENCFF137WCM.tar.gz https://www.encodeproject.org/files/ENCFF137WCM/@@download/ENCFF137WCM.tar.gz
    !cd /content/ref && mkdir -p hepg2_models && tar -xzf ENCFF137WCM.tar.gz -C hepg2_models
folds = sorted(glob.glob('/content/ref/hepg2_models/**/*chrombpnet_nobias*.h5', recursive=True))
print(len(folds), 'HepG2 folds')

bench = load_labeled_variants('/content/RegLens/data/benchmarks/kircher_mpra_grch38.tsv')
hep_scorer = ChromBPNetScorer(KerasChromBPNetBackend(folds, average_rc=True),
                              window_length=2114, model_name='HepG2-5fold+rc')
t0 = time.time()
hep = evaluate_batched(bench, hep_scorer, genome_path='/content/ref/hg38.fa', progress=True)
print(f'HepG2 scored in {time.time()-t0:.0f}s')
HEPG2 = {e: a for e, a, *_ in hep.per_source_auroc() if a is not None}

In [ ]:
# --- Crossover summary + double-dissociation test + bootstrap CI ------------
from reglens.validation.lineage import (crossover_summary, is_double_dissociation,
                                         bootstrap_crossover_ci)
from reglens.report.plot import plot_crossover
per = {'K562': K562, 'HepG2': HEPG2}
s = crossover_summary(per)
print('crossover means:', s)
print('DOUBLE DISSOCIATION:', is_double_dissociation(s, 'K562', 'HepG2'))
for comp, d in bootstrap_crossover_ci(per, 'K562', 'HepG2').items():
    print(f"  {comp:14s} Δ=+{d['delta']:.3f}  95% CI [{d['lo']:+.3f},{d['hi']:+.3f}]  "
          f"p(wrong sign)={d['p_wrong_sign']:.2f}")
plot_crossover(per, '/content/crossover.png')
from IPython.display import Image; Image('/content/crossover.png')

## Honest reading

The dissociation is **asymmetric**: robust on the hematopoietic side (K562 +0.147, CI
clears zero) and directional-but-not-significant on the hepatic side (HepG2 +0.030, CI
crosses zero). The compartment means flip in both directions and the per-element extremes
(SORT1 up, PKLR-48h 0.805→0.505 down) are decisive. See `RESULTS.md` for the full table.